#### Primeiro, vamos analizar o enunciado do problema:
2- Uma planta química necessita de etileno a 325 K e 15 bar. Entretanto, o etileno disponível
encontra-se a 480 K e 50 bar. Para reduzir a temperatura e a pressão do etileno até os valores
desejados, foram propostas duas configurações de processo.
- Configuração I: O etileno passa inicialmente por uma válvula de estrangulamento, na qual sua
pressão é reduzida de 50 bar para 15 bar. Em seguida, é enviado a um trocador de calor para
reduzir sua temperatura até 325 K.
- Configuração II: O etileno é inicialmente expandido em uma turbina adiabática e reversível até
15 bar e, posteriormente, enviado a um trocador de calor.

Admitindo que o etileno obedeça à equação de estado de Peng-Robinson, determine a
transferência de calor e o trabalho por unidade de massa de etileno em cada uma das
configurações propostas.

Vamos começar montando a primeira configuração:


Temos os seguintes valores: 
- T1 = 480 K 
- T2 = ? 
- T3 = 325 K
- P1 = 50 bar
- P2 = 15 bar
- P3 = 15 bar

Queremos saber a transferência de calor e trabalho em cada uma das confiuações propostas

Vamos separar o processo em duas partes. Primeiramente, vamos trabalhar com a válvula de estrangulamento. 

A equação do balanço de energia pode ser descrita pela expressão:
- $\Delta$ U = m $\Delta$ H<sub>1-2</sub> + Q + W

Hipóteses: 

1 - Não há acúmulo de energia ($\Delta$ U = 0)

2 - Não há realizaçõ de trabalho de eixo, compressão ou expansão (W=0)

3 - Não há troca de calor no processo (Q = 0)

Logo, podemos chegar na expressão: 
- m $\Delta$ H<sub>1-2</sub> = 0

Logo, o processo é isoentálpico. Dessa forma, podemos determinar a temperatura T2 a partir do cálculo da variação de entalpia.

In [1]:
from substancias import etileno
from eos import PR
from propriedades import entropia, entalpia
from balancos import balanco_energia, balanco_entropia
from constantes import BAR, kJ
from solvers import TporH, TporS

In [2]:
T1 = 480 
T3 = 325
P1 = 50 * BAR
P2 = 15 * BAR
P3 = P2
n = 1

De acordo com as propriedades residuais, a entalpia pode ser dada pela equação: 
- H<sub>2</sub> -H<sub>1</sub> = - H<sup>R</sup><sub>1</sub> + H<sup>id</sup><sub>2</sub> - H<sup>id</sup><sub>1</sub> + H<sup>R</sup><sub>2</sub> = 0

No entanto, repare que Não possuímos o valor da temperatura no estágio de saída da válvula. Dessa forma, usaremos um módulo construído para fazer as operações:

Tf = eos, T1, P1, P2, n, dH_alvo, T_min, T_max, fase

Assim como esperado, há um erro caso tentemos usar a temperatura entre o valor da temperatura crítica e da temperatura inicial, pois em uma expansão na válvula há diminuição de temperatura.

eos = PR( etileno )

Tf = TporH( eos, T1, P1, P2, n, 0, etileno.tc, 325, "vapor")

print(f"Temperatura final: {Tf} K") 

ValueError: A função deve ter sinais opostos em a e b.

Reparemos também que a temperatura do estágio inicial é superior a temperatura crítica da própria substância (305,2 K). Desssa forma, podemos estimar que existe uma raíz entre o valor da temperatura crítica e a temperatura inicial do processo. Assim:

In [3]:
eos = PR( etileno )
T2 = TporH( eos, T1, P1, P2, 1, 0, etileno.tc, 480, "vapor")
print(f"Temperatura final: {T2} K") 

Temperatura final: 468.3690738708526 K


Agora que temos a temperatura T2, podemos fazer um balanço de energia no trocador de calor: 
- $\Delta$ U = m $\Delta$ H<sub>1-2</sub> + Q + W

Hipóteses: 

1 - Não há acúmulo de energia ($\Delta$ U = 0)

2 - Não há realizaçõ de trabalho de eixo, compressão ou expansão (W=0)

Logo, podemos chegar na expressão: 

m $\Delta$ H<sub>2-1</sub> = Q


De acordo com as propriedades residuais, a entalpia pode ser dada pela equação: 
- H<sub>2</sub> -H<sub>1</sub> = - H<sup>R</sup><sub>1</sub> + H<sup>id</sup><sub>2</sub> - H<sup>id</sup><sub>1</sub> + H<sup>R</sup><sub>2</sub> 

In [4]:
Hres1 = entalpia().residual(eos, T2, P2, n, "vapor")
Hres2 = entalpia().residual(eos, T3, P3, n, "vapor")
Hideal = entalpia().ideal(etileno, T2, T3, P2, P3, n)
Htotal = -Hres1 + Hideal + Hres2
print(f"-Hres1: {Hres1} + Hideal: {Hideal} + Hres2: {Hres2} = Htotal: {Htotal}")

-Hres1: -333.6250039318803 + Hideal: -7715.701921361201 + Hres2: -635.0546301770004 = Htotal: -8017.131547606321


In [5]:
Htotal = -Htotal #Devemos inverter o sinal porque na função definimos delta H = Hentrada - Hsaída
Q = balanco_energia(0, Htotal, None, 0)
print(f"Q: {Q} J/mol")

Q: -8017.131547606321 J/mol


Logo, Q = 8017.131547606321 J/mol. Mas queremos em base mássica, logo:

In [6]:
Qmass = etileno.Jmol_to_Jkg(Q)
print(f"Q = {Qmass} J/kg")

Q = -285774.9892210138 J/kg


In [7]:
Qmass = Qmass/kJ
print(f"Q = {Qmass} kJ/kg")

Q = -285.7749892210138 kJ/kg


Logo, os valores de Q e W da segunda configuração correspondem a: 
- Q = -285.7749892210138 kJ/kg 
- W = 0